In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_splitfrom sklearn.linear_model import LinearRegressionfrom sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeaturesfrom sklearn.metrics import mean_squared_error, r2_scoreimport warningswarnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r'D:\shubham\House prediction ML\Bengaluru_House_Data.csv')print("Data loaded successfully")print(df.shape)print(df.head())print(df.info())

In [ ]:
def extract_bhk(size):    if pd.isna(size):        return np.nan    size = str(size).lower()    if "bhk" in size or "bedroom" in size:        try:            return int("".join(filter(str.isdigit, size)))        except:            return np.nan    return np.nandef clean_sqft(x):    if isinstance(x, str):        if "-" in x:            try:                low, high = map(float, x.split("-"))                return (low + high) / 2            except:                return np.nan        try:            return float(x)        except:            return np.nan    return float(x)df["bhk"] = df["size"].apply(extract_bhk)df["total_sqft"] = df["total_sqft"].apply(clean_sqft)df["bath"] = pd.to_numeric(df["bath"], errors="coerce")df["balcony"] = pd.to_numeric(df["balcony"], errors="coerce").fillna(0)df["has_balcony"] = (df["balcony"] > 0).astype(int)print(df[["size", "bhk", "total_sqft", "bath", "balcony"]].head())

In [ ]:
print("Before cleaning:", df.shape)df = df.dropna(subset=["bhk", "total_sqft", "bath", "price"])df = df[(df["price"] < 200) & (df["bhk"] < 20) & (df["bath"] < 15) & (df["total_sqft"] > 300) & (df["total_sqft"] < 30000)]print("After cleaning:", df.shape)df["sqft_per_bhk"] = df["total_sqft"] / df["bhk"]df["bath_per_bhk"] = df["bath"] / df["bhk"]df["location"] = df["location"].astype(str).str.strip()print(df[["bhk", "total_sqft", "bath", "price", "sqft_per_bhk", "bath_per_bhk"]].head())

In [ ]:
top_locations = df["location"].value_counts().head(100).indexX = df[["location", "total_sqft", "bhk", "bath", "has_balcony", "sqft_per_bhk", "bath_per_bhk"]].copy()X["location"] = X["location"].apply(lambda x: x if x in top_locations else "other")y = df["price"]print("Feature matrix and target ready")print(X.shape, y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)print("Train/test split:", X_train.shape, X_test.shape)if "sparse_output" in OneHotEncoder.__init__.__code__.co_varnames:    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)else:    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)location_train = ohe.fit_transform(X_train[["location"]])location_test = ohe.transform(X_test[["location"]])num_features = ["total_sqft", "bhk", "bath", "has_balcony", "sqft_per_bhk", "bath_per_bhk"]poly = PolynomialFeatures(degree=2, include_bias=False)X_train_num = poly.fit_transform(X_train[num_features])X_test_num = poly.transform(X_test[num_features])scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train_num)X_test_scaled = scaler.transform(X_test_num)X_train_scaled = np.hstack([X_train_scaled, location_train])X_test_scaled = np.hstack([X_test_scaled, location_test])print("Prepared numeric and categorical features")print("Training feature shape:", X_train_scaled.shape)

In [ ]:
model = LinearRegression()model.fit(X_train_scaled, y_train)y_pred_train = model.predict(X_train_scaled)y_pred_test = model.predict(X_test_scaled)print("Linear regression model trained")

In [ ]:
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))train_r2 = r2_score(y_train, y_pred_train)test_r2 = r2_score(y_test, y_pred_test)print(f"TRAIN RMSE: {train_rmse:.2f}, TRAIN R²: {train_r2:.4f}")print(f"TEST RMSE: {test_rmse:.2f}, TEST R²: {test_r2:.4f}")

In [ ]:
def predict_house_price(location, total_sqft, bhk, bath, balcony=0):    input_df = pd.DataFrame({        "location": [location],        "total_sqft": [total_sqft],        "bhk": [bhk],        "bath": [bath],        "has_balcony": [1 if balcony > 0 else 0]    })    input_df["sqft_per_bhk"] = input_df["total_sqft"] / input_df["bhk"]    input_df["bath_per_bhk"] = input_df["bath"] / input_df["bhk"]    input_df["location"] = input_df["location"].apply(lambda x: x if x in top_locations else "other")    location_onehot = ohe.transform(input_df[["location"]])    numeric_values = poly.transform(input_df[["total_sqft", "bhk", "bath", "has_balcony", "sqft_per_bhk", "bath_per_bhk"]])    numeric_scaled = scaler.transform(numeric_values)    input_scaled = np.hstack([numeric_scaled, location_onehot])    return model.predict(input_scaled)[0]print("Prediction helper ready")

In [ ]:
print(predict_house_price("Whitefield", 1500, 3, 2, 1))print(predict_house_price("Electronic City Phase II", 1000, 2, 2, 0))print(predict_house_price("Koramangala", 3000, 4, 4, 2))